[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LSDtopotools/lsdtt3_notebooks/blob/main/chi_analysis/granite_creek_chi_concavity_knickpoints.ipynb)

# Chi analysis of Granite Creek, Hells Canyon, Idaho: concavity, k<sub>sn</sub> and knickpoints

Granite Creek drains the Seven Devils Mountains in western Idaho and joins the Snake River in Hells Canyon. It falls about 2,400 m in under 20 km, from the peaks around He Devil to the river, which makes it a good place to look for changes in channel steepness.

The workflow is:
1. Install `lsdviztools3` (the plotting package, from PyPI) and the `lsdtt3` command-line tools (from the latest GitHub release).
2. Download a **10 m** USGS 3DEP DEM with `lsdtt-fetch-raster`.
3. Make a hillshade with `lsdtt-raster-preprocessing`.
4. Find the **concavity index** (m/n) of the basin with the **disorder method** in `lsdtt-chi-analysis`: test a range of m/n values and keep the one with the least disorder.
5. Rerun `lsdtt-chi-analysis` at that m/n to get the **channel steepness index** k<sub>sn</sub>, and map it.
6. Find the **knickpoints**, the places where k<sub>sn</sub> changes, and plot the major ones.

Each `lsdtt3` program is controlled by a small **parameter file**: a text file of `key: value` lines. We write these files from Python and then run the programs with `!`.

## Setup

### First set up condacolab

This step takes around 2 minutes. **`condacolab.install()` restarts the Colab runtime.** You will see a message that the session crashed: this is expected. Wait for it to reconnect, then carry on with the next cell.

In [ ]:
!pip install -q condacolab

In [ ]:
import condacolab
condacolab.install()

### Install PyGMT

`lsdviztools3` draws its maps with PyGMT, which needs the GMT library. pip cannot install GMT, so we install PyGMT (and GMT with it) from conda-forge. This is the slowest step and takes around a minute.

In [ ]:
# mamba prints hundreds of lines, so we hide them and only show the end of
# the output if something goes wrong.
import subprocess

result = subprocess.run(["mamba", "install", "-y", "pygmt"], capture_output=True, text=True)
if result.returncode == 0:
    print("pygmt installed successfully.")
else:
    print("pygmt installation FAILED. The last lines of the output were:\n")
    print("\n".join((result.stdout + result.stderr).splitlines()[-30:]))

### Install lsdviztools3 from PyPI

`lsdviztools3` needs Python 3.12 or later; the next cell prints the version this runtime has.

The `lsdviz3-*` commands below start with `MPLBACKEND=Agg`. Colab sets `MPLBACKEND` for its own inline plots, the commands inherit it, and matplotlib then fails to start in them. `Agg` is matplotlib's backend for writing image files, which is all the commands do.

In [ ]:
import sys
print(sys.version)
!pip install -q lsdviztools3
!MPLBACKEND=Agg lsdviz3-chi --help | head -n 3

### Get the lsdtt3 command-line tools

The Linux package of the latest release. `releases/latest/download/` always points at the newest version.

In [ ]:
!wget -q https://github.com/LSDtopotools/lsdtt3/releases/latest/download/lsdtt3-backend-linux-x86_64-core.tar.gz
!tar -xzf lsdtt3-backend-linux-x86_64-core.tar.gz

Now we tell the system where to find the `lsdtt3` programs and the libraries they need. If your runtime restarts later, run this cell again.

In [ ]:
import glob
import os

root = sorted(glob.glob("/content/lsdtt3-backend-linux-x86_64-core-v*"))[-1]
print("Using", root)
os.environ["PATH"] = f"{root}/bin:" + os.environ["PATH"]
os.environ["LD_LIBRARY_PATH"] = f"{root}/lib:" + os.environ.get("LD_LIBRARY_PATH", "")

# Do NOT set GDAL_DATA / PROJ_DATA / PROJ_LIB here. The lsdtt3 programs find
# their own bundled PROJ and GDAL data, and pointing the Python session at the
# backend's proj.db breaks rasterio and pyproj, which lsdviztools3 uses.

Check that it works: this prints the version of the tools.

In [ ]:
!lsdtt-chi-analysis -v

## Step 1: Download a 10 m DEM

`lsdtt-fetch-raster` downloads a DEM for a box given in longitude and latitude. Here we use

* `dem_source: us_3dep`: the USGS 3D Elevation Program seamless DEM. It is a **bare-earth terrain model**, covers the United States, needs no account or key, and its default product is **10 m** (1/3 arc-second). We do not set `grid_spacing`, so the output stays at the native 10 m.
* We do not set `target_epsg` either: the tool picks the UTM zone from the centre of the box (here UTM zone 11N), so distances are in metres.

The box contains the whole of the Granite Creek basin, about 86 km², with a margin. **The whole basin must be inside the DEM**: a basin cut by the edge of the DEM is missing some of its drainage area, so `lsdtt-chi-analysis` drops it.

The output is `granite_DEM.tif`.

In [ ]:
with open("granite_fetch.param", "w") as f:
    f.write("write_fname: granite\n")
    f.write("west: -116.68\n")
    f.write("east: -116.53\n")
    f.write("south: 45.17\n")
    f.write("north: 45.37\n")
    f.write("dem_source: us_3dep\n")   # USGS 3DEP, 10 m bare earth

!lsdtt-fetch-raster ./ granite_fetch.param
!ls -lh granite*

## Step 2: Make a hillshade

`lsdtt-raster-preprocessing` writes `granite_hs_hillshade.tif`, lit from the northwest. The Snake River runs along the western edge of the box; Granite Creek comes in from the south-east, from the Seven Devils.

In [ ]:
with open("granite_hillshade.param", "w") as f:
    f.write("read_fname: granite_DEM.tif\n")
    f.write("write_fname: granite_hs\n")
    f.write("write_output_format: gtiff\n")
    f.write("write_hillshade_raster: true\n")

!lsdtt-raster-preprocessing ./ granite_hillshade.param

In [ ]:
from IPython.display import Image, display

from lsdviztools3.render.hillshade import render_hillshade
from lsdviztools3.render.style import MapStyle
from lsdviztools3.render.base import save_figure

hs_style = MapStyle(cmap="gray", transparent=False)
fig = render_hillshade("granite_hs_hillshade.tif", coordinates="local", shade=False, style=hs_style)
save_figure(fig, "granite_hillshade.png", style=hs_style)
display(Image("granite_hillshade.png", width=600))

## Step 3: The basin outlet

`lsdtt-chi-analysis` takes the basin outlet from a CSV file with `latitude` and `longitude` columns (WGS84, as copied from Google Maps). The point below is on Granite Creek about 550 m above its mouth, so it cannot be mistaken for the Snake River.

The outlet is **snapped** onto the channel network: `expanding_square` takes the nearest channel pixel within `basin_outlet_snap_max_search_radius_pixels` (5 pixels, 50 m here).

### Choosing the channel network

Chi is computed on channel pixels only. A channel starts where the upslope area reaches `threshold_contributing_pixels`. This is a **count of pixels**, so the same area needs a different number at each resolution: on a 10 m DEM, **2000 pixels = 0.2 km²**.

In [ ]:
import pandas as pd

pd.DataFrame({"latitude": [45.3438], "longitude": [-116.6516]}).to_csv("granite_outlet.csv", index=False)

# The settings shared by both chi runs below
common = {
    "read_fname": "granite_DEM.tif",
    "basin_outlet_fname": "granite_outlet.csv",
    "basin_outlet_snapping_method": "expanding_square",
    "basin_outlet_snap_max_search_radius_pixels": 5,
    "threshold_contributing_pixels": 2000,     # 0.2 km2 on a 10 m DEM
    "n_largest_basins": 1,
    "fit_chi_networks": "true",
    "fit_critical_divergence_metres": 5,       # see Step 5
}


def write_param(fname, settings):
    """Write a dictionary of settings as an lsdtt3 parameter file."""
    with open(fname, "w") as f:
        for key, value in settings.items():
            f.write(f"{key}: {value}\n")

## Step 4: The best concavity from a disorder scan

### How it works (Mudd et al., 2018)

Chi (χ) is a coordinate along each channel that integrates drainage area upstream from the outlet:

```
chi = integral from the outlet of (A0 / A)^(m/n) dx
```

where A is drainage area, A0 a reference area (1 m²) and **m/n the concavity index**. If the basin is in steady state, with uniform uplift and erodibility, and m/n is right, **every channel in the basin collapses onto one line** in a plot of elevation against chi. With the wrong m/n the tributaries sit above or below the main stem.

The **disorder** measures how far the channels are from collapsing onto one curve. Sort all the channel pixels by elevation; if chi only ever increases with elevation, the channels are perfectly collapsed. Every reversal adds to the disorder. lsdtt3 computes it for each m/n value in a range (`mn_min` to `mn_max` in steps of `mn_step`); the m/n with the **lowest disorder** is the best estimate.

### How certain is it?

A basin's disorder depends on which tributaries it happens to have. With `run_bootstrap: true` the tool resamples the side branches (always keeping the main stem) `bootstrap_replicates` times and recomputes the disorder at every m/n. That gives a mean and a 95 % interval for each m/n. An m/n value whose interval overlaps that of the best one cannot be told apart from it.

This run writes `granite_disorder_chi_disorder_sweep.csv`: one row per m/n value. It takes a minute or two.

In [ ]:
sweep_settings = dict(common)
sweep_settings.update({
    "write_fname": "granite_disorder",
    "mn_min": 0.15,
    "mn_max": 0.85,
    "mn_step": 0.05,
    "run_bootstrap": "true",
    "bootstrap_replicates": 200,
    # Only the disorder is needed from this run, not the per-m/n fit tables
    "write_fit_tables": "false",
    "write_chi_tributary_table": "false",
    "write_chi_knickpoint_table": "false",
    "write_ksn_basin_summary": "false",
})
write_param("granite_disorder.param", sweep_settings)

!lsdtt-chi-analysis ./ granite_disorder.param

In [ ]:
sweep = pd.read_csv("granite_disorder_chi_disorder_sweep.csv")
sweep["m_over_n"] = sweep["m_over_n"].round(3)

best = sweep.loc[sweep["full_network_disorder"].idxmin()]
best_mn = float(best["m_over_n"])

# m/n values whose bootstrap interval overlaps that of the best m/n
overlap = sweep[(sweep["bootstrap_percentile_2p5"] <= best["bootstrap_percentile_97p5"]) &
                (sweep["bootstrap_percentile_97p5"] >= best["bootstrap_percentile_2p5"])]

print(f"Best m/n (lowest disorder): {best_mn}")
print(f"m/n values not distinguishable from it: {overlap['m_over_n'].min()} to {overlap['m_over_n'].max()}")
sweep[["m_over_n", "full_network_disorder", "bootstrap_mean",
       "bootstrap_percentile_2p5", "bootstrap_percentile_97p5", "n_side_branches"]]

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.fill_between(sweep["m_over_n"], sweep["bootstrap_percentile_2p5"], sweep["bootstrap_percentile_97p5"],
                color="tab:blue", alpha=0.2, label="bootstrap 95 % interval")
ax.plot(sweep["m_over_n"], sweep["bootstrap_mean"], "--", color="tab:blue", label="bootstrap mean")
ax.plot(sweep["m_over_n"], sweep["full_network_disorder"], "o-", color="k", label="whole network")
ax.axvline(best_mn, color="tab:red", lw=1)
ax.text(best_mn, ax.get_ylim()[1], f"  best m/n = {best_mn}", color="tab:red", va="top")
ax.set_xlabel("m/n")
ax.set_ylabel("Disorder")
ax.set_title("Granite Creek: disorder against concavity index")
ax.legend()
plt.show()

**Question:** is there a clear minimum, or is the curve flat over a wide range of m/n? A flat curve means the data cannot pin down the concavity well. The method also assumes uniform uplift and erodibility: where the rock type changes along the channels, no m/n collapses them onto one curve.

## Step 5: k<sub>sn</sub> at the best concavity

Now we run `lsdtt-chi-analysis` again at the best m/n only (`m_over_n`). It fits each channel with straight segments in chi–elevation space; the gradient of each segment is its **normalised channel steepness index** k<sub>sn</sub> (with A0 = 1 m², the units are m<sup>2m/n</sup>).

The fit splits a channel wherever the data are more than `fit_critical_divergence_metres` from the line. 5 m suits a 10 m DEM: smaller values give more, shorter segments that follow the noise in the DEM; larger values keep only the big changes.

This run also writes the fitted segments as lines (`write_chi_fit_segment_lines`) for the maps, and the knickpoint tables used in Step 6.

In [ ]:
mn_suffix = f"{best_mn:.3f}".replace(".", "p")   # how lsdtt3 writes m/n in file names, e.g. 0p450

ksn_settings = dict(common)
ksn_settings.update({
    "write_fname": "granite_ksn",
    "m_over_n": best_mn,
    "write_chi_fit_segment_lines": "true",
    "write_vector_format": "flatgeobuf",
    "write_chi_knickpoint_table": "true",
})
write_param("granite_ksn.param", ksn_settings)

!lsdtt-chi-analysis ./ granite_ksn.param
!ls granite_ksn_chi_*mn_{mn_suffix}*

`lsdviz3-chi` draws the fitted network coloured by k<sub>sn</sub> over a hillshade, and the chi–elevation profile of the basin with the fitted segments.

In [ ]:
!MPLBACKEND=Agg lsdviz3-chi --data-dir . --prefix granite_ksn --out-prefix granite --m-over-n {best_mn} \
    --dem granite_DEM.tif --out-dir . --per-basin

for png in ["granite_ksn_map.png"] + sorted(glob.glob("granite_chi*.png")):
    display(Image(png, width=700))

In [ ]:
segments = pd.read_csv(f"granite_ksn_chi_fit_segments_mn_{mn_suffix}.csv")
print(f"{len(segments)} segments")
segments["k_sn"].describe().round(1)

## Step 6: Knickpoints

A **knickpoint candidate** is any place where two fitted segments meet. `granite_ksn_chi_knickpoints_mn_*.csv` lists them, with

| Column | Meaning |
|---|---|
| `ksn_upstream`, `ksn_downstream` | k<sub>sn</sub> of the segments above and below |
| `delta_ksn` | upstream minus downstream: **positive where the channel steepens upstream** |
| `log_ratio_ksn` | log10 of their ratio |
| `at_junction` | 1 at a tributary mouth, where the tributary meets its parent channel |
| `chi`, `elevation`, `flow_distance_m`, `drainage_area_m2`, `x`, `y` | where it is |

Every segment boundary is a candidate, so most are small. The tool does not pick the major ones for you: `granite_ksn_chi_knickpoint_thresholds_mn_*.csv` **suggests thresholds** from the distribution of |`delta_ksn`| and |`log_ratio_ksn`|: the 90th and 95th percentiles (`p90`, `p95`), the median plus three median absolute deviations (`median_plus_3mad`), and Otsu's threshold (`otsu`), which best splits the values into two groups. `n_above` is how many candidates each one keeps.

In [ ]:
knickpoints = pd.read_csv(f"granite_ksn_chi_knickpoints_mn_{mn_suffix}.csv")
thresholds = pd.read_csv(f"granite_ksn_chi_knickpoint_thresholds_mn_{mn_suffix}.csv")
print(f"{len(knickpoints)} knickpoint candidates, {int(knickpoints['at_junction'].sum())} of them at tributary mouths")
thresholds

`lsdviz3-knickpoints` picks the major knickpoints with one of these thresholds (`--threshold`, here the 95th percentile of |`delta_ksn`|) and draws:

* the distribution of the k<sub>sn</sub> changes, with the suggested thresholds and the one chosen;
* the major knickpoints on the map, over the k<sub>sn</sub> network;
* the major knickpoints on the chi–elevation profile;
* where they sit along chi.

It also writes the selected knickpoints to `granite_major_knickpoints.csv`. Changes at tributary mouths are left out unless you add `--include-junctions`.

In [ ]:
!MPLBACKEND=Agg lsdviz3-knickpoints --data-dir . --prefix granite_ksn --out-prefix granite --m-over-n {best_mn} \
    --dem granite_DEM.tif --out-dir . --threshold p95

for png in ["granite_knickpoint_distribution.png", "granite_knickpoint_map.png",
            "granite_knickpoint_profile.png", "granite_knickpoints_along_chi.png"]:
    if os.path.exists(png):
        display(Image(png, width=700))

In [ ]:
major = pd.read_csv("granite_major_knickpoints.csv")
cols = [c for c in ["elevation", "flow_distance_m", "drainage_area_m2", "ksn_downstream",
                    "ksn_upstream", "delta_ksn", "chi"] if c in major.columns]
major.sort_values("elevation")[cols].round(1)

**Questions:**

* Are the major knickpoints at similar **elevations** on different channels? Knickpoints that migrate upstream from one change in base level tend to sit at similar elevations, or similar chi, across a basin.
* Do they steepen upstream (`delta_ksn` > 0) or shallow upstream? Compare their positions with the hillshade: some may be where the channels leave the glacially carved valleys of the Seven Devils.

## Things to try

* Change `--threshold` to `otsu` or `median_plus_3mad`, or give a number. How many knickpoints survive each?
* Use `--measure log_ratio` to select by the ratio of k<sub>sn</sub> rather than the difference: a change from 50 to 100 then counts as much as one from 200 to 400.
* Change `fit_critical_divergence_metres` (for example to 3 or 10) in Step 5. How do the segments and knickpoints change?
* Change `threshold_contributing_pixels` to 5000 (0.5 km²). Does the best m/n change?
* Narrow the disorder scan around the best value with a smaller `mn_step`, for example 0.3 to 0.6 in steps of 0.02.
* Add more Hells Canyon tributaries as extra rows of `granite_outlet.csv` (and enlarge the box so it contains all their basins; set `n_largest_basins` to the number of outlets). Do they share a concavity?

## Every run is recorded

Each program writes `<write_fname>_ingested_params.param`: every parameter it used. You can run it again as it is to repeat the run. Keep these files with your results.

## References

* Perron, J. T. and Royden, L. (2013). An integral approach to bedrock river profile analysis. *Earth Surface Processes and Landforms*, 38(6), 570–576. https://doi.org/10.1002/esp.3302
* Mudd, S. M., Clubb, F. J., Gailleton, B. and Hurst, M. D. (2018). How concave are river channels? *Earth Surface Dynamics*, 6, 505–523. https://doi.org/10.5194/esurf-6-505-2018
* Hergarten, S., Robl, J. and Stüwe, K. (2016). Tectonic geomorphology at small catchment sizes – extensions of the stream-power approach and the χ method. *Earth Surface Dynamics*, 4(1), 1–9. https://doi.org/10.5194/esurf-4-1-2016
* The lsdtt3 documentation of the method: https://lsdtopotools.github.io/lsdtt3/design_and_algorithms/chi_analysis.html